# Single-Head Self-Attention in PyTorch

In this notebook, we build a single-head self-attention forward pass using dummy data.

By the end, we will understand how:
- tokens are projected into queries, keys, and values,
- attention scores are computed,
- softmax turns scores into weights,
- and weighted values produce the final output.

In [1]:
import math
import torch

import torch.nn as nn

## Step 1: Create Input

We will create some fake data for this example.

Think of it like this:

- `batch_size = 2` means we have 2 sentences.
- `seq_len = 4` means each sentence has 4 tokens.
- `d_model = 8` means each token is represented by an 8-dimensional vector.
- `d_k` is the size of the vectors used to compare tokens in attention.

So, `x` can be thought of as:

```text
[
  [word1, word2, word3, word4],  # sentence 1
  [word1, word2, word3, word4]   # sentence 2
]
```

Also, each word is represented by a vector of size 8.

In [2]:
torch.manual_seed(42)
batch_size = 2
seq_len = 4
d_model = 8
d_k = 8

x = torch.randn(batch_size, seq_len, d_model)


## Step 2: Create Weight Matrices

We prepare three different views of each word:

| Matrix | Purpose |
|---|---|
| `W_q` | Helps form the query, or the “question” a token asks. |
| `W_k` | Helps form the key, which is used to match queries. |
| `W_v` | Holds the actual information that will be combined later. |

Why do we need three matrices?

Because attention works like this:

- a token creates a **query**,
- compares it with other tokens’ **keys**,
- and then uses the corresponding **values** to build the output.

In short:  
“I ask a question (`Q`), compare it with other tokens’ keys (`K`), and then take their values (`V`).”

In [3]:
W_q = torch.randn(d_model, d_k)
W_k = torch.randn(d_model, d_k)
W_v = torch.randn(d_model, d_k)

## Step 3: Create Q, K, and V

Each word is transformed into three vectors:

- a query (`Q`)
- a key (`K`)
- a value (`V`)

We get them by multiplying the input `x` with three different weight matrices:

```python
Q = x @ W_q
K = x @ W_k
V = x @ W_v
```

These vectors are used in the next step to compute attention scores.

In [4]:
Q = x @ W_q
K = x @ W_k
V = x @ W_v

## Step 4: Compare Words (Compute Attention Scores)

Now we want to answer this question:

> How much should this word pay attention to every other word?

We do this by taking the dot product of `Q` and the transpose of `K`:

```python
scores = Q @ K.T
```

For one sentence, let’s ignore the batch dimension for now.

- `Q` has shape `(4, 8)`
- `K` has shape `(4, 8)`

After transposing `K`:

- `K.T` has shape `(8, 4)`

So the score matrix becomes:

- `scores = Q @ K.T`
- `scores` has shape `(4, 4)`

What does this `4 x 4` matrix mean?

- Each row represents one word.
- Each column represents another word.

```text
        w1     w2     w3     w4
w1   [ s11   s12   s13   s14 ]
w2   [ s21   s22   s23   s24 ]
w3   [ s31   s32   s33   s34 ]
w4   [ s41   s42   s43   s44 ]
```

For example:

- `s13` = how much word 1 cares about word 3
- `s22` = how much word 2 cares about itself

So, this matrix shows how much each word should attend to every other word.

In [5]:
scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)

## Step 5: Convert Scores to Probabilities

Next, we turn the raw attention scores into probabilities:

```python
attn_weights = torch.softmax(scores, dim=-1)
```

What is happening here?

We convert the raw scores into normalized weights, so each row tells us how much attention to give to each token.

For example:

- Before softmax: `[2.1, 0.5, 1.2, 3.0]`
- After softmax: `[0.2, 0.05, 0.1, 0.65]`

Now each row sums to 1

Meaning, for `word1`:

- 20% attention to `w1`
- 5% attention to `w2`
- 10% attention to `w3`
- 65% attention to `w4`

In [6]:
attn_weights = torch.softmax(scores, dim=-1)

## Step 6: Combine the Values

Now we use the attention weights to combine the `V` vectors.

Let’s focus on one word first: `word1`.

The first row of the attention matrix tells us how much `word1` takes from each word:

```text
[a11, a12, a13, a14]
```

This means:

- `a11` = how much word 1 cares about itself
- `a12` = how much word 1 cares about word 2
- `a13` = how much word 1 cares about word 3
- `a14` = how much word 1 cares about word 4

Now we combine those weights with the value vectors:

```text
output_w1 = a11 * V1 + a12 * V2 + a13 * V3 + a14 * V4
```

So, the output for `word1` is a weighted mixture of all the value vectors.

We do the same for `word2`:

```text
output_w2 = a21 * V1 + a22 * V2 + a23 * V3 + a24 * V4
```

And in general, for any word `i`:

```text
output_wi = ai1 * V1 + ai2 * V2 + ai3 * V3 + ai4 * V4
```

So, each output word becomes a blend of all words, based on how much attention it gives to each one.

In [7]:
output = attn_weights @ V

In [8]:
print("===== Single-Head Attention Demo =====")
print("Input shape:", x.shape)
print("Q shape:", Q.shape)
print("K shape:", K.shape)
print("V shape:", V.shape)
print("Attention scores shape:", scores.shape)
print("Attention weights shape:", attn_weights.shape)
print("Output shape:", output.shape)
print("Attention weights for batch 0:\n", attn_weights[0])
print()

===== Single-Head Attention Demo =====
Input shape: torch.Size([2, 4, 8])
Q shape: torch.Size([2, 4, 8])
K shape: torch.Size([2, 4, 8])
V shape: torch.Size([2, 4, 8])
Attention scores shape: torch.Size([2, 4, 4])
Attention weights shape: torch.Size([2, 4, 4])
Output shape: torch.Size([2, 4, 8])
Attention weights for batch 0:
 tensor([[1.0000e+00, 4.4039e-06, 3.4055e-11, 2.1570e-08],
        [9.9883e-01, 1.1604e-03, 1.4906e-08, 6.8233e-06],
        [2.5128e-03, 1.1434e-04, 2.7579e-04, 9.9710e-01],
        [1.4965e-04, 7.4394e-05, 2.7356e-03, 9.9704e-01]])



### Interpreting the Attention Weights

Each row in the attention matrix shows how one token distributes its attention across all tokens in the sentence.

Large values mean strong attention, and small values mean weak attention.

Here, the first token focuses mostly on itself, while the third and fourth tokens focus mainly on `w4`.